In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


# Metacognitive Calibration Benchmark

Tests whether a model's stated confidence matches actual accuracy.
Grounded in Nelson & Narens (1990) metamemory monitoring framework.

**Cognitive Science Basis**: Retrospective confidence judgment.
**Human baseline ECE**: 0.10–0.20
**Score**: 1 - ECE (higher = better calibrated)

## MetaCog Benchmark 1: Retrospective Confidence Calibration

Tests whether a model's stated confidence in its answers correlates
with its actual accuracy. Well-calibrated models should be right ~80%
of the time when they say they're 80% confident.

### Cognitive Science Basis

- Based on the metacognitive monitoring framework (Nelson & Narens, 1990)
- Measures "retrospective confidence" — post-answer confidence ratings
- Uses Expected Calibration Error (ECE) as the primary metric
- Human baseline ECE is typically 0.10–0.20

### Methodology

1. Present diverse questions across domains and difficulty levels
2. Ask model to answer AND rate confidence (0–100)
3. Bin answers by confidence level
4. Compare stated confidence to actual accuracy per bin
5. Compute ECE = weighted average of |accuracy_bin - confidence_bin|
Score: Brier Skill Score (BSS = 1 - BS/BS_ref), which rewards both
calibration AND resolution (discrimination). Unlike 1-ECE, BSS properly
penalizes always-uncertain strategies and rewards models that assign high
confidence to correct answers and low confidence to incorrect ones.
BS_ref = climatological baseline (base_rate * (1 - base_rate)).

### Shortcut Resistance

- Questions span many domains (no single-domain memorisation helps)
- Mix of difficulty levels forces genuine uncertainty
- Confidence must be stated alongside the answer (no post-hoc adjustment)


In [ ]:
import kaggle_benchmarks as kbench
from dataclasses import dataclass
import numpy as np
import pandas as pd
import re
import json
# ─── Question Dataset ───────────────────────────────────────────────
# Import from data module (includes both handcrafted + procedural questions)
CALIBRATION_QUESTIONS = [{'accept_patterns': ['Au'],
  'answer': 'Au',
  'difficulty': 1,
  'domain': 'chemistry',
  'question': 'What is the chemical symbol for gold?'},
 {'accept_patterns': ['6', 'six'],
  'answer': '6',
  'difficulty': 1,
  'domain': 'math',
  'question': 'How many sides does a hexagon have?'},
 {'accept_patterns': ['Mars'],
  'answer': 'Mars',
  'difficulty': 1,
  'domain': 'astronomy',
  'question': 'What planet is known as the Red Planet?'},
 {'accept_patterns': ['1945'],
  'answer': '1945',
  'difficulty': 1,
  'domain': 'history',
  'question': 'In which year did World War II end?'},
 {'accept_patterns': ['Tokyo'],
  'answer': 'Tokyo',
  'difficulty': 1,
  'domain': 'geography',
  'question': 'What is the capital of Japan?'},
 {'accept_patterns': ['deoxyribonucleic acid'],
  'answer': 'deoxyribonucleic acid',
  'difficulty': 1,
  'domain': 'biology',
  'question': 'What does DNA stand for?'},
 {'accept_patterns': ['3', 'three'],
  'answer': '3',
  'difficulty': 1,
  'domain': 'biology',
  'question': 'How many hearts does an octopus have?'},
 {'accept_patterns': ['Vatican', 'Vatican City'],
  'answer': 'Vatican City',
  'difficulty': 1,
  'domain': 'geography',
  'question': 'What is the smallest country in the world by area?'},
 {'accept_patterns': ['2001'],
  'answer': '2001',
  'difficulty': 1,
  'domain': 'technology',
  'question': 'In what year was Wikipedia launched?'},
 {'accept_patterns': ['skin'],
  'answer': 'skin',
  'difficulty': 1,
  'domain': 'biology',
  'question': 'What is the largest organ in the human body?'},
 {'accept_patterns': ['Shakespeare'],
  'answer': 'William Shakespeare',
  'difficulty': 1,
  'domain': 'literature',
  'question': "Who wrote the play 'Romeo and Juliet'?"},
 {'accept_patterns': ['100'],
  'answer': '100',
  'difficulty': 1,
  'domain': 'physics',
  'question': 'What is the boiling point of water in degrees Celsius at standard atmospheric '
              'pressure?'},
 {'accept_patterns': ['206'],
  'answer': '206',
  'difficulty': 1,
  'domain': 'anatomy',
  'question': 'How many bones are in the adult human body?'},
 {'accept_patterns': ['343'],
  'answer': '343',
  'difficulty': 1,
  'domain': 'physics',
  'question': 'What is the speed of sound in air at 20°C, in meters per second?'},
 {'accept_patterns': ['New Zealand'],
  'answer': 'New Zealand',
  'difficulty': 1,
  'domain': 'history',
  'question': 'Which country was the first to grant women the right to vote in national '
              'elections?'},
 {'accept_patterns': ['5730', '5,730'],
  'answer': '5730',
  'difficulty': 2,
  'domain': 'physics',
  'question': 'What is the half-life of Carbon-14, approximately in years?'},
 {'accept_patterns': ['1494'],
  'answer': '1494',
  'difficulty': 2,
  'domain': 'history',
  'question': 'In what year was the Treaty of Tordesillas signed, dividing the New World between '
              'Spain and Portugal?'},
 {'accept_patterns': ['osmium', 'Os'],
  'answer': 'osmium',
  'difficulty': 2,
  'domain': 'chemistry',
  'question': 'What is the densest naturally occurring element?'},
 {'accept_patterns': ['11', 'eleven'],
  'answer': '11',
  'difficulty': 2,
  'domain': 'geography',
  'question': 'How many time zones does Russia span?'},
 {'accept_patterns': ['tungsten', 'W', 'wolfram'],
  'answer': 'tungsten',
  'difficulty': 2,
  'domain': 'chemistry',
  'question': 'What element has the highest melting point?'},
 {'accept_patterns': ['1971'],
  'answer': '1971',
  'difficulty': 2,
  'domain': 'technology',
  'question': 'In what year was the first network email sent by Ray Tomlinson?'},
 {'accept_patterns': ['5', 'five'],
  'answer': '5',
  'difficulty': 2,
  'domain': 'geography',
  'question': 'How many US states border the Pacific Ocean?'},
 {'accept_patterns': ['7', 'seven'],
  'answer': '7',
  'difficulty': 2,
  'domain': 'geology',
  'question': 'What is the Mohs hardness of quartz?'},
 {'accept_patterns': ['1989'],
  'answer': '1989',
  'difficulty': 2,
  'domain': 'history',
  'question': 'In what year did the Berlin Wall fall?'},
 {'accept_patterns': ['6', 'six'],
  'answer': '6',
  'difficulty': 2,
  'domain': 'literature',
  'question': 'How many completed novels did Jane Austen write?'},
 {'accept_patterns': ['1066'],
  'answer': '1066',
  'difficulty': 2,
  'domain': 'history',
  'question': 'In what year was the Battle of Hastings fought?'},
 {'accept_patterns': ['828', '829.8'],
  'answer': '828',
  'difficulty': 2,
  'domain': 'architecture',
  'question': 'What is the exact height of the Burj Khalifa in meters (to the tip)?'},
 {'accept_patterns': ['54'],
  'answer': '54',
  'difficulty': 2,
  'domain': 'geography',
  'question': 'How many recognized countries are in Africa according to the United Nations?'},
 {'accept_patterns': ['1768'],
  'answer': '1768',
  'difficulty': 2,
  'domain': 'history',
  'question': 'In what year was the first edition of the Encyclopaedia Britannica published?'},
 {'accept_patterns': ['101325', '101,325'],
  'answer': '101325',
  'difficulty': 2,
  'domain': 'physics',
  'question': 'What is the standard atmospheric pressure at sea level in pascals?'},
 {'accept_patterns': ['2.5', '3', '2.5%', '3%'],
  'answer': '2.5',
  'difficulty': 3,
  'domain': 'earth science',
  'question': "What percentage of Earth's water is fresh water (not salt water)? Give to one "
              'decimal place.'},
 {'accept_patterns': ['Antarctica'],
  'answer': 'Antarctica',
  'difficulty': 3,
  'domain': 'geography',
  'question': 'What is the driest continent on Earth by average annual precipitation?'},
 {'accept_patterns': ['Sweden'],
  'answer': 'Sweden',
  'difficulty': 3,
  'domain': 'geography',
  'question': 'Which country has the most islands in the world?'},
 {'accept_patterns': ['No', 'no', 'not visible', 'cannot'],
  'answer': 'No',
  'difficulty': 3,
  'domain': 'science',
  'question': 'Is the Great Wall of China visible to the naked eye from low Earth orbit?'},
 {'accept_patterns': ['0.007297', '0.00730', '1/137'],
  'answer': '0.007297',
  'difficulty': 3,
  'domain': 'physics',
  'question': 'What is the value of the fine-structure constant (alpha) to 4 significant figures? '
              'Express as a decimal.'},
 {'accept_patterns': ['8', 'eight'],
  'answer': '8',
  'difficulty': 3,
  'domain': 'history',
  'question': 'How many U.S. presidents have died while in office (including assassinations)?'},
 {'accept_patterns': ['Saturn'],
  'answer': 'Saturn',
  'difficulty': 3,
  'domain': 'astronomy',
  'question': 'Which planet in our solar system currently has the most known moons?'},
 {'accept_patterns': ['1826', '1827'],
  'answer': '1826',
  'difficulty': 3,
  'domain': 'history',
  'question': 'In what year was the earliest surviving photograph (by Nicéphore Niépce) taken?'},
 {'accept_patterns': ['6.62607015', '6.626 070 15'],
  'answer': '6.62607015e-34',
  'difficulty': 3,
  'domain': 'physics',
  'question': 'What is the exact value of the Planck constant h in J·s, as defined in the 2019 SI? '
              'Give all significant digits.'},
 {'accept_patterns': ['0.05', '$0.05', '5 cents', 'five cents'],
  'answer': '0.05',
  'difficulty': 3,
  'domain': 'math',
  'question': 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How '
              'much does the ball cost in dollars?'},
 {'accept_patterns': ['24'],
  'answer': '24',
  'difficulty': 3,
  'domain': 'math',
  'question': 'If you have a 4x4x4 cube made of 64 small unit cubes, and you paint the outside, '
              'how many unit cubes have exactly two painted faces?'},
 {'accept_patterns': ['1955', '1956'],
  'answer': '1955',
  'difficulty': 3,
  'domain': 'biology',
  'question': 'In what year were human chromosomes correctly counted as 46 (not 48)?'},
 {'accept_patterns': ['5050', '5,050'],
  'answer': '5050',
  'difficulty': 3,
  'domain': 'math',
  'question': 'What is the sum of all integers from 1 to 100?'},
 {'accept_patterns': ['2/3', '0.667', '0.66', '66.7%'],
  'answer': '2/3',
  'difficulty': 3,
  'domain': 'math',
  'question': 'In the original Monty Hall problem, what is the probability of winning if you '
              'switch doors? Express as a fraction.'},
 {'accept_patterns': ['37'],
  'answer': '37',
  'difficulty': 3,
  'domain': 'literature',
  'question': 'How many plays are in the traditional Shakespeare canon (First Folio plus '
              'Pericles)?'},
 {'accept_patterns': ['6.02214076'],
  'answer': '6.02214076e23',
  'difficulty': 3,
  'domain': 'chemistry',
  'question': 'What is the exact value of the Avogadro constant as defined in the 2019 SI '
              'redefinition, in mol⁻¹?'},
 {'accept_patterns': ['1742'],
  'answer': '1742',
  'difficulty': 3,
  'domain': 'history of science',
  'question': 'In what year did Anders Celsius propose his temperature scale?'},
 {'accept_patterns': ['3422', '3,422', '3410', '3414'],
  'answer': '3422',
  'difficulty': 3,
  'domain': 'chemistry',
  'question': 'What is the melting point of tungsten in degrees Celsius, rounded to the nearest '
              'degree?'},
 {'accept_patterns': ['30'],
  'answer': '30',
  'difficulty': 3,
  'domain': 'math',
  'question': 'How many edges does an icosahedron have?'},
 {'accept_patterns': ['1.380649'],
  'answer': '1.380649e-23',
  'difficulty': 3,
  'domain': 'physics',
  'question': 'What is the exact value of the Boltzmann constant k in J/K as defined in the 2019 '
              'SI?'},
 {'accept_patterns': ['5', 'five'],
  'answer': '5',
  'difficulty': 4,
  'domain': 'abstract algebra',
  'question': 'How many groups of order 8 exist up to isomorphism? (Counting all groups of order '
              '8.)'},
 {'accept_patterns': ['50', '50%', '51'],
  'answer': '50',
  'difficulty': 4,
  'domain': 'probability',
  'question': 'In a room of 23 people, what is the probability that at least two share a birthday? '
              'Give as a percentage rounded to the nearest whole number.'},
 {'accept_patterns': ['40000', '40,000', '39000', '39,000', '41000', '41,000'],
  'answer': '40000',
  'difficulty': 4,
  'domain': 'geography',
  'question': 'What was the population of Liechtenstein in 2024, to the nearest thousand?'},
 {'accept_patterns': ['Q', 'q'],
  'answer': 'Q',
  'difficulty': 4,
  'domain': 'trivia',
  'question': 'What is the only letter that does not appear in the name of any US state?'},
 {'accept_patterns': ['21'],
  'answer': '21',
  'difficulty': 4,
  'domain': 'math',
  'question': 'How many two-digit prime numbers are there?'},
 {'accept_patterns': ['615.75', '615.8', '196*pi', '196π'],
  'answer': '615.75',
  'difficulty': 4,
  'domain': 'math',
  'question': 'What is the surface area of a sphere with radius 7, in terms of exact value? Give a '
              'numerical answer rounded to 2 decimal places.'},
 {'accept_patterns': ['649740', '649,740'],
  'answer': '649740',
  'difficulty': 4,
  'domain': 'probability',
  'question': 'In a standard 52-card deck, what is the probability of being dealt a royal flush in '
              "5-card poker? Express as '1 in N' where N is the answer."},
 {'accept_patterns': ['no missing dollar',
                      'there is no missing',
                      'misdirection',
                      'accounting error',
                      'fallacy',
                      'includes'],
  'answer': 'There is no missing dollar. The $27 paid includes the $25 for the room plus the $2 '
            'the bellboy kept. The $29 figure incorrectly adds cost and tip.',
  'difficulty': 4,
  'domain': 'logic',
  'question': 'Three people check into a hotel room that costs $30. They each pay $10. The manager '
              'realizes the room should only cost $25, so he gives $5 to the bellboy to return. '
              'The bellboy keeps $2 and gives back $1 to each guest. Now each guest has paid $9 '
              '(total $27), plus the bellboy has $2, totaling $29. Where is the missing dollar?'},
 {'accept_patterns': ['108'],
  'answer': '108',
  'difficulty': 4,
  'domain': 'chemistry',
  'question': 'What is the atomic number of the element Hassium?'},
 {'accept_patterns': ['1689'],
  'answer': '1689',
  'difficulty': 4,
  'domain': 'history',
  'question': 'In what year was the Treaty of Nerchinsk signed between Russia and the Qing '
              'Dynasty?'},
 {'accept_patterns': ['5'],
  'answer': '5',
  'difficulty': 4,
  'domain': 'math',
  'question': 'What is the 10th digit of pi after the decimal point?'},
 {'accept_patterns': ['52'],
  'answer': '52',
  'difficulty': 4,
  'domain': 'math',
  'question': 'How many perfect numbers are known to exist as of 2024?'},
 {'accept_patterns': ['440000', '440,000', '439804', '439,804'],
  'answer': '440000',
  'difficulty': 4,
  'domain': 'math',
  'question': 'If you fold a standard piece of paper (0.1mm thick) in half 42 times, approximately '
              'how thick would it be? Answer in kilometers to the nearest thousand.'},
 {'accept_patterns': ['RuBisCO',
                      'rubisco',
                      'ribulose-1,5-bisphosphate carboxylase',
                      'ribulose bisphosphate carboxylase'],
  'answer': 'RuBisCO',
  'difficulty': 4,
  'domain': 'biochemistry',
  'question': 'What is the name of the enzyme that catalyzes the conversion of carbon dioxide and '
              'water into glucose during the Calvin cycle in photosynthesis?'},
 {'accept_patterns': ['3', 'three'],
  'answer': '3',
  'difficulty': 4,
  'domain': 'logic',
  'question': 'If you have 12 identical-looking balls, one of which is either heavier or lighter '
              'than the rest, what is the minimum number of weighings on a balance scale needed to '
              'identify the odd ball and determine if it is heavier or lighter?'},
 {'accept_patterns': ['330', '325', '340'],
  'answer': '330',
  'difficulty': 5,
  'domain': 'history',
  'question': 'What is the exact year the Kingdom of Aksum (Axum) converted to Christianity under '
              'King Ezana?'},
 {'accept_patterns': ['22.59', '22.587'],
  'answer': '22.59',
  'difficulty': 5,
  'domain': 'chemistry',
  'question': 'What is the density of osmium in g/cm³, to 2 decimal places?'},
 {'accept_patterns': ['1928'],
  'answer': '1928',
  'difficulty': 5,
  'domain': 'history',
  'question': 'In what year was the Oxford English Dictionary first fully published (all volumes '
              'of the first edition)?'},
 {'accept_patterns': ['16'],
  'answer': '16',
  'difficulty': 5,
  'domain': 'math',
  'question': 'How many prime numbers are there between 1000 and 1100?'},
 {'accept_patterns': ['0.44', '0.49'],
  'answer': '0.44',
  'difficulty': 5,
  'domain': 'geography',
  'question': 'What is the exact area of Vatican City in square kilometers, to 2 decimal places?'},
 {'accept_patterns': ['23', 'Article 23'],
  'answer': '23',
  'difficulty': 5,
  'domain': 'law',
  'question': 'What specific article number of the UN Charter establishes the Security Council?'},
 {'accept_patterns': ['299792458', '299,792,458'],
  'answer': '299792458',
  'difficulty': 5,
  'domain': 'physics',
  'question': 'What is the speed of light in vacuum to 9 significant figures in m/s?'},
 {'accept_patterns': ['1956'],
  'answer': '1956',
  'difficulty': 5,
  'domain': 'biology',
  'question': 'In what year did Tjio and Levan publish their paper correctly establishing the '
              'human chromosome number as 46?'},
 {'accept_patterns': ['49/20', '2.45'],
  'answer': '49/20',
  'difficulty': 5,
  'domain': 'math',
  'question': 'What is the sum of the reciprocals of all positive integers from 1 to 6, expressed '
              'as a fraction in lowest terms?'},
 {'accept_patterns': ['June 30, 1908', '30 June 1908', 'June 30 1908', '1908-06-30'],
  'answer': 'June 30, 1908',
  'difficulty': 5,
  'domain': 'history',
  'question': 'What was the exact date (day, month, year) of the Tunguska event?'},
 {'accept_patterns': ['52'],
  'answer': '52',
  'difficulty': 5,
  'domain': 'math',
  'question': 'How many known Mersenne primes exist as of 2024?'},
 {'accept_patterns': ['38', '38 minutes', '45'],
  'answer': '38',
  'difficulty': 5,
  'domain': 'history',
  'question': 'What is the shortest war in recorded history (between Britain and Zanzibar)? How '
              'many minutes did it last?'},
 {'accept_patterns': ['87 BC', '87 BCE', '100 BC', '150 BC', '205 BC'],
  'answer': '87 BC',
  'difficulty': 5,
  'domain': 'history',
  'question': 'What specific year was the Antikythera mechanism estimated to have been built (the '
              'commonly cited date)?'},
 {'accept_patterns': ['4'],
  'answer': '4',
  'difficulty': 5,
  'domain': 'math',
  'question': "What is the 100th decimal digit of the mathematical constant e (Euler's number)?"},
 {'accept_patterns': ['Reiwa', '令和'],
  'answer': 'Reiwa',
  'difficulty': 5,
  'domain': 'culture',
  'question': 'What is the name of the Japanese era (nengō) that began on May 1, 2019?'},
 {'accept_patterns': ['117'],
  'answer': '117',
  'difficulty': 1,
  'domain': 'arithmetic',
  'question': 'What is 92 + 25?'},
 {'accept_patterns': ['78'],
  'answer': '78',
  'difficulty': 1,
  'domain': 'arithmetic',
  'question': 'What is 13 × 6?'},
 {'accept_patterns': ['20000'],
  'answer': '20000',
  'difficulty': 1,
  'domain': 'conversion',
  'question': 'How many meters are in 20 kilometers?'},
 {'accept_patterns': ['3'],
  'answer': '3',
  'difficulty': 1,
  'domain': 'conversion',
  'question': 'How many hours are in 180 minutes?'},
 {'accept_patterns': ['25'],
  'answer': '25',
  'difficulty': 1,
  'domain': 'arithmetic',
  'question': 'What is 5 squared?'},
 {'accept_patterns': ['121'],
  'answer': '121',
  'difficulty': 1,
  'domain': 'arithmetic',
  'question': 'What is 24 + 97?'},
 {'accept_patterns': ['138'],
  'answer': '138',
  'difficulty': 1,
  'domain': 'arithmetic',
  'question': 'What is 46 × 3?'},
 {'accept_patterns': ['50000'],
  'answer': '50000',
  'difficulty': 1,
  'domain': 'conversion',
  'question': 'How many meters are in 50 kilometers?'},
 {'accept_patterns': ['2'],
  'answer': '2',
  'difficulty': 1,
  'domain': 'conversion',
  'question': 'How many hours are in 120 minutes?'},
 {'accept_patterns': ['9'],
  'answer': '9',
  'difficulty': 1,
  'domain': 'arithmetic',
  'question': 'What is 3 squared?'},
 {'accept_patterns': ['-4'],
  'answer': '-4',
  'difficulty': 2,
  'domain': 'algebra',
  'question': 'Solve for x: 3x + 8 = -4'},
 {'accept_patterns': ['241.5'],
  'answer': '241.5',
  'difficulty': 2,
  'domain': 'geometry',
  'numeric_tolerance': 0.01,
  'question': 'What is the area of a triangle with base 21 cm and height 23 cm?'},
 {'accept_patterns': ['15.75'],
  'answer': '15.75',
  'difficulty': 2,
  'domain': 'arithmetic',
  'numeric_tolerance': 0.01,
  'question': 'What is 25% of 63?'},
 {'accept_patterns': ['500'],
  'answer': '500',
  'difficulty': 2,
  'domain': 'physics',
  'numeric_tolerance': 0.01,
  'question': 'A car travels at 100 km/h for 5 hours. How far does it go (in km)?'},
 {'accept_patterns': ['1'],
  'answer': '1',
  'difficulty': 2,
  'domain': 'arithmetic',
  'question': 'What is the remainder when 818 is divided by 19?'},
 {'accept_patterns': ['-3'],
  'answer': '-3',
  'difficulty': 2,
  'domain': 'algebra',
  'question': 'Solve for x: 8x + 15 = -9'},
 {'accept_patterns': ['138'],
  'answer': '138',
  'difficulty': 2,
  'domain': 'geometry',
  'numeric_tolerance': 0.01,
  'question': 'What is the area of a triangle with base 23 cm and height 12 cm?'},
 {'accept_patterns': ['46.4'],
  'answer': '46.4',
  'difficulty': 2,
  'domain': 'arithmetic',
  'numeric_tolerance': 0.01,
  'question': 'What is 10% of 464?'},
 {'accept_patterns': ['200'],
  'answer': '200',
  'difficulty': 2,
  'domain': 'physics',
  'numeric_tolerance': 0.01,
  'question': 'A car travels at 40 km/h for 5 hours. How far does it go (in km)?'},
 {'accept_patterns': ['12'],
  'answer': '12',
  'difficulty': 2,
  'domain': 'arithmetic',
  'question': 'What is the remainder when 532 is divided by 13?'},
 {'accept_patterns': ['-6'],
  'answer': '-6',
  'difficulty': 2,
  'domain': 'algebra',
  'question': 'Solve for x: 6x + 7 = -29'},
 {'accept_patterns': ['203'],
  'answer': '203',
  'difficulty': 2,
  'domain': 'geometry',
  'numeric_tolerance': 0.01,
  'question': 'What is the area of a triangle with base 29 cm and height 14 cm?'},
 {'accept_patterns': ['15.3'],
  'answer': '15.3',
  'difficulty': 2,
  'domain': 'arithmetic',
  'numeric_tolerance': 0.01,
  'question': 'What is 15% of 102?'},
 {'accept_patterns': ['90'],
  'answer': '90',
  'difficulty': 2,
  'domain': 'physics',
  'numeric_tolerance': 0.01,
  'question': 'A car travels at 60 km/h for 1.5 hours. How far does it go (in km)?'},
 {'accept_patterns': ['12'],
  'answer': '12',
  'difficulty': 2,
  'domain': 'arithmetic',
  'question': 'What is the remainder when 467 is divided by 13?'},
 {'accept_patterns': ['192'],
  'answer': '192',
  'difficulty': 3,
  'domain': 'math',
  'question': 'What is the sum of the first 8 terms of the arithmetic sequence starting at 10 with '
              'common difference 4?'},
 {'accept_patterns': ['120'],
  'answer': '120',
  'difficulty': 3,
  'domain': 'combinatorics',
  'question': 'How many ways can you choose 3 items from 10 distinct items (combinations)?'},
 {'accept_patterns': ['-5'],
  'answer': '-5',
  'difficulty': 3,
  'domain': 'algebra',
  'question': 'Find the roots of x² + 1x - 20 = 0. Give the smaller root.'},
 {'accept_patterns': ['2'],
  'answer': '2',
  'difficulty': 3,
  'domain': 'math',
  'question': 'What is the greatest common divisor (GCD) of 90 and 332?'},
 {'accept_patterns': ['50.16'],
  'answer': '50.16',
  'difficulty': 3,
  'domain': 'arithmetic',
  'numeric_tolerance': 0.02,
  'question': 'An item costs $57. It is discounted by 20%, then 10% tax is added. What is the '
              'final price?'},
 {'accept_patterns': ['288'],
  'answer': '288',
  'difficulty': 3,
  'domain': 'math',
  'question': 'What is the sum of the first 9 terms of the arithmetic sequence starting at 4 with '
              'common difference 7?'},
 {'accept_patterns': ['5'],
  'answer': '5',
  'difficulty': 3,
  'domain': 'combinatorics',
  'question': 'How many ways can you choose 4 items from 5 distinct items (combinations)?'},
 {'accept_patterns': ['-1'],
  'answer': '-1',
  'difficulty': 3,
  'domain': 'algebra',
  'question': 'Find the roots of x² x - 1 = 0. Give the smaller root.'},
 {'accept_patterns': ['1'],
  'answer': '1',
  'difficulty': 3,
  'domain': 'math',
  'question': 'What is the greatest common divisor (GCD) of 90 and 487?'},
 {'accept_patterns': ['47.63'],
  'answer': '47.63',
  'difficulty': 3,
  'domain': 'arithmetic',
  'numeric_tolerance': 0.02,
  'question': 'An item costs $49. It is discounted by 10%, then 8% tax is added. What is the final '
              'price?'},
 {'accept_patterns': ['455'],
  'answer': '455',
  'difficulty': 3,
  'domain': 'math',
  'question': 'What is the sum of the first 13 terms of the arithmetic sequence starting at 5 with '
              'common difference 5?'},
 {'accept_patterns': ['20'],
  'answer': '20',
  'difficulty': 3,
  'domain': 'combinatorics',
  'question': 'How many ways can you choose 3 items from 6 distinct items (combinations)?'},
 {'accept_patterns': ['-2'],
  'answer': '-2',
  'difficulty': 3,
  'domain': 'algebra',
  'question': 'Find the roots of x² - 1x - 6 = 0. Give the smaller root.'},
 {'accept_patterns': ['3'],
  'answer': '3',
  'difficulty': 3,
  'domain': 'math',
  'question': 'What is the greatest common divisor (GCD) of 393 and 186?'},
 {'accept_patterns': ['27.12'],
  'answer': '27.12',
  'difficulty': 3,
  'domain': 'arithmetic',
  'numeric_tolerance': 0.02,
  'question': 'An item costs $29. It is discounted by 15%, then 10% tax is added. What is the '
              'final price?'}]
_UNUSED_INLINE_QUESTIONS = [
    # TIER 1: Easy
    {"question": "What is the chemical symbol for gold?", "answer": "Au", "domain": "chemistry", "difficulty": 1},
    {"question": "How many sides does a hexagon have?", "answer": "6", "domain": "math", "difficulty": 1},
    {"question": "What planet is known as the Red Planet?", "answer": "Mars", "domain": "astronomy", "difficulty": 1},
    {"question": "What is the largest organ in the human body?", "answer": "skin", "domain": "biology", "difficulty": 1},
    {"question": "In which year did World War II end?", "answer": "1945", "domain": "history", "difficulty": 1},
    {"question": "What is the boiling point of water in degrees Celsius at standard atmospheric pressure?", "answer": "100", "domain": "physics", "difficulty": 1},
    {"question": "Who wrote the play 'Romeo and Juliet'?", "answer": "Shakespeare", "domain": "literature", "difficulty": 1},
    {"question": "What is the capital of Japan?", "answer": "Tokyo", "domain": "geography", "difficulty": 1},
    {"question": "What does DNA stand for?", "answer": "deoxyribonucleic acid", "domain": "biology", "difficulty": 1},
    {"question": "What is the speed of light in a vacuum, approximately in km/s?", "answer": "300000", "domain": "physics", "difficulty": 1},
    # TIER 2: Medium
    {"question": "What is the smallest prime number greater than 50?", "answer": "53", "domain": "math", "difficulty": 2},
    {"question": "Which enzyme is primarily responsible for unwinding the DNA double helix during replication?", "answer": "helicase", "domain": "biology", "difficulty": 2},
    {"question": "In what year was the Treaty of Westphalia signed, ending the Thirty Years' War?", "answer": "1648", "domain": "history", "difficulty": 2},
    {"question": "What is the derivative of ln(x) with respect to x?", "answer": "1/x", "domain": "math", "difficulty": 2},
    {"question": "Which country has the longest coastline in the world?", "answer": "Canada", "domain": "geography", "difficulty": 2},
    {"question": "What is the half-life of Carbon-14, approximately in years?", "answer": "5730", "domain": "physics", "difficulty": 2},
    {"question": "Who composed 'The Four Seasons'?", "answer": "Vivaldi", "domain": "music", "difficulty": 2},
    {"question": "What is the Mohs hardness of quartz?", "answer": "7", "domain": "geology", "difficulty": 2},
    {"question": "In computing, what does the acronym RISC stand for?", "answer": "reduced instruction set computer", "domain": "computing", "difficulty": 2},
    {"question": "What neurotransmitter is most directly associated with the reward system in the brain?", "answer": "dopamine", "domain": "neuroscience", "difficulty": 2},
    {"question": "What is the approximate distance from Earth to the Moon in kilometers?", "answer": "384400", "domain": "astronomy", "difficulty": 2},
    {"question": "Which philosopher wrote 'Critique of Pure Reason'?", "answer": "Kant", "domain": "philosophy", "difficulty": 2},
    {"question": "What is the oxidation state of iron in rust (Fe2O3)?", "answer": "+3", "domain": "chemistry", "difficulty": 2},
    {"question": "In what year did the Berlin Wall fall?", "answer": "1989", "domain": "history", "difficulty": 2},
    {"question": "What is the name of the longest river in Africa?", "answer": "Nile", "domain": "geography", "difficulty": 2},
    # TIER 3: Hard
    {"question": "What is the sum of the first 20 prime numbers?", "answer": "639", "domain": "math", "difficulty": 3},
    {"question": "In which specific year did the Tunguska event occur?", "answer": "1908", "domain": "history", "difficulty": 3},
    {"question": "What is the atomic number of Promethium?", "answer": "61", "domain": "chemistry", "difficulty": 3},
    {"question": "How many bones are in the adult human wrist (carpal bones only)?", "answer": "8", "domain": "anatomy", "difficulty": 3},
    {"question": "What is the escape velocity from the surface of Mars in km/s, approximately?", "answer": "5.0", "domain": "physics", "difficulty": 3},
    {"question": "Who proved the incompleteness theorems in 1931?", "answer": "Gödel", "domain": "math", "difficulty": 3},
    {"question": "What is the name of the deepest known point in the Earth's oceans?", "answer": "Challenger Deep", "domain": "geography", "difficulty": 3},
    {"question": "In which year was the Rosetta Stone discovered?", "answer": "1799", "domain": "history", "difficulty": 3},
    {"question": "What is the second most abundant element in the Earth's crust by mass?", "answer": "silicon", "domain": "geology", "difficulty": 3},
    {"question": "What is the value of the golden ratio (phi) to 3 decimal places?", "answer": "1.618", "domain": "math", "difficulty": 3},
    {"question": "Which organelle is known as the 'powerhouse of the cell' and uses oxidative phosphorylation?", "answer": "mitochondria", "domain": "biology", "difficulty": 3},
    {"question": "What is the coefficient of restitution for a perfectly elastic collision?", "answer": "1", "domain": "physics", "difficulty": 3},
    {"question": "Who formulated the Church-Turing thesis alongside Turing?", "answer": "Church", "domain": "computing", "difficulty": 3},
    {"question": "In what year was the Universal Declaration of Human Rights adopted?", "answer": "1948", "domain": "history", "difficulty": 3},
    {"question": "What is the name of the largest known structure in the observable universe?", "answer": "Hercules-Corona Borealis Great Wall", "domain": "astronomy", "difficulty": 3},
]
# ─── Structured Output Schema ──────────────────────────────────────
@dataclass
class ConfidentAnswer:
    """Model's answer with a confidence rating."""
    answer: str           # The actual answer to the question
    confidence: int       # 0-100 confidence rating
# ─── Answer Verification ────────────────────────────────────────────
def normalize(text: str) -> str:
    """Normalize text for fuzzy matching."""
    text = text.lower().strip()
    # Remove articles, punctuation
    text = re.sub(r'\b(the|a|an)\b', '', text)
    text = re.sub(r'[^\w\s\.\+\-/]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
def check_answer(model_answer: str, correct_answer: str) -> bool:
    """Check if model's answer matches the correct answer (fuzzy)."""
    model_norm = normalize(model_answer)
    correct_norm = normalize(correct_answer)
    # Direct containment (either direction)
    if correct_norm in model_norm or model_norm in correct_norm:
        return True
    # For numeric answers, try numeric comparison
    try:
        model_num = float(re.search(r'[\-\+]?[\d\.]+', model_norm).group())
        correct_num = float(re.search(r'[\-\+]?[\d\.]+', correct_norm).group())
        # Allow 5% tolerance for approximate numeric answers
        if correct_num == 0:
            return abs(model_num) < 0.01
        return abs(model_num - correct_num) / abs(correct_num) < 0.05
    except (ValueError, AttributeError, ZeroDivisionError):
        pass
    return False
# ─── Scoring Functions ──────────────────────────────────────────────
def brier_skill_score(confidences_0_100: list, outcomes_binary: list) -> float:
    """
    Brier Skill Score: BSS = 1 - BS / BS_ref
    BS = mean((forecast - outcome)^2)  — Brier Score
    BS_ref = base_rate * (1 - base_rate) — climatological baseline
    Rewards BOTH calibration (confidence matches accuracy) AND resolution
    (ability to discriminate correct from incorrect answers).
    Unlike 1-ECE, an always-uncertain strategy scores ~0 rather than ~1.
    Returns: float in (-inf, 1]. Clamped to [0, 1] for benchmark scoring.
      - BSS > 0: better than climatological baseline
      - BSS = 0: equivalent to always predicting base rate
      - BSS < 0: worse than baseline (overconfident or anti-correlated)
    """
    conf = np.array(confidences_0_100) / 100.0
    out = np.array(outcomes_binary, dtype=float)
    BS = float(np.mean((conf - out) ** 2))
    base_rate = float(out.mean())
    BS_ref = base_rate * (1 - base_rate)
    # Degenerate case: all outcomes identical → use uniform (0.5) reference
    if BS_ref < 1e-10:
        BS_ref = float(np.mean((0.5 - out) ** 2))
    if BS_ref < 1e-10:
        return 0.0
    return 1.0 - BS / BS_ref
def compute_ece(confidences_0_100: list, accuracies: list, n_bins: int = 10) -> dict:
    """
    Compute Expected Calibration Error (diagnostic only — not used in final score).
    Returns dict with:
    - ece: float (0-1, lower = better calibrated)
    - bin_data: list of dicts with bin details
    - n_samples: int
    """
    confidences = np.array(confidences_0_100) / 100.0  # Normalize to 0-1
    accuracies = np.array(accuracies, dtype=float)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_data = []
    ece = 0.0
    total = len(confidences)
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        bin_count = mask.sum()
        if bin_count == 0:
            bin_data.append({
                "bin": f"{lo:.1f}-{hi:.1f}",
                "count": 0,
                "avg_confidence": None,
                "avg_accuracy": None,
                "gap": None,
            })
            continue
        avg_conf = confidences[mask].mean()
        avg_acc = accuracies[mask].mean()
        gap = abs(avg_acc - avg_conf)
        ece += (bin_count / total) * gap
        bin_data.append({
            "bin": f"{lo:.1f}-{hi:.1f}",
            "count": int(bin_count),
            "avg_confidence": round(float(avg_conf), 3),
            "avg_accuracy": round(float(avg_acc), 3),
            "gap": round(float(gap), 3),
        })
    return {
        "ece": round(float(ece), 4),
        "bin_data": bin_data,
        "n_samples": total,
    }
# ─── The Benchmark Task ────────────────────────────────────────────
@kbench.task(name="Retrospective Calibration")
def metacog_calibration(llm) -> float:
    """
    Metacognitive Calibration Benchmark.
    Measures how well a model's stated confidence matches its actual accuracy.
    Score = composite:
      0.50 × extreme_accuracy^1.5 — amplified extreme-question accuracy (difficulty >= 4)
      0.25 × BSS-normalized — (bss_raw + 1) / 2, calibration quality
      0.25 × uncertainty awareness — 1 - mean_confidence_on_hard_incorrect items
    This composite differentiates models that are somewhat overconfident from those
    that are wildly overconfident, unlike BSS which clamps all poor calibrators to 0.
    Cognitive Science Basis: Nelson & Narens (1990) metamemory monitoring framework.
    Lichtenstein et al. (1982) calibration; Murphy (1973) skill score decomposition.
    """
    confidences = []
    accuracies = []
    results_log = []
    for i, q in enumerate(CALIBRATION_QUESTIONS):
        with kbench.chats.new(f"q_{i}"):
            prompt = (
                f"Answer the following question. After your answer, rate your confidence "
                f"from 0 to 100 (where 0 = pure guess, 100 = absolutely certain).\n\n"
                f"Question: {q['question']}\n\n"
                f"Respond with ONLY a JSON object in this exact format:\n"
                f'{{"answer": "<your answer>", "confidence": <0-100>}}'
            )
            try:
                result = llm.prompt(prompt, schema=ConfidentAnswer)
                answer = result.answer
                confidence = max(0, min(100, result.confidence))
            except Exception:
                # Fallback: try to parse raw text
                raw = llm.prompt(prompt)
                try:
                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
                    answer = str(parsed.get("answer", ""))
                    confidence = int(parsed.get("confidence", 50))
                    confidence = max(0, min(100, confidence))
                except Exception:
                    answer = raw
                    confidence = 50  # Default if parsing fails
            is_correct = check_answer(answer, q["answer"])
            confidences.append(confidence)
            accuracies.append(is_correct)
            results_log.append({
                "question": q["question"],
                "correct_answer": q["answer"],
                "model_answer": answer,
                "confidence": confidence,
                "is_correct": is_correct,
                "domain": q["domain"],
                "difficulty": q["difficulty"],
            })
    # Compute scoring metrics
    bss_raw = brier_skill_score(confidences, accuracies)
    metrics = compute_ece(confidences, accuracies)
    # --- Component 1: Calibration (1 - ECE) ---
    calibration_score = 1.0 - metrics['ece']
    # --- BSS normalized ---
    # Maps BSS from [-1,1] range to [0,1]: perfectly calibrated=1, perfectly anti-calibrated=0
    # Preserves variance without clamping negatives to the same floor
    bss_normalized = max(0.0, min(1.0, (bss_raw + 1.0) / 2.0))
    # --- Component 2: Confidence discrimination ---
    conf_correct = [c for c, a in zip(confidences, accuracies) if a]
    conf_incorrect = [c for c, a in zip(confidences, accuracies) if not a]
    if conf_correct and conf_incorrect:
        discrimination = (np.mean(conf_correct) - np.mean(conf_incorrect)) / 100.0
        discrimination = max(0.0, min(1.0, discrimination))
    elif conf_correct and not conf_incorrect:
        discrimination = 1.0
    else:
        discrimination = 0.0
    # --- Extreme-question accuracy (difficulty >= 4 only) ---
    extreme_items = [r for r in results_log if r['difficulty'] >= 4]
    if extreme_items:
        extreme_correct = sum(1 for r in extreme_items if r['is_correct'])
        extreme_accuracy = extreme_correct / len(extreme_items)
    else:
        extreme_accuracy = 0.0
    # --- Uncertainty awareness on hard incorrect items ---
    hard_incorrect = [r for r in results_log if r['difficulty'] >= 3 and not r['is_correct']]
    if hard_incorrect:
        hard_overconfidence = np.mean([r['confidence'] for r in hard_incorrect]) / 100.0
        uncertainty_awareness = 1.0 - hard_overconfidence
    else:
        uncertainty_awareness = 1.0
    # --- Composite score ---
    # ext^1.5 amplifies accuracy differences: 0.5->0.354, 0.7->0.586, 0.9->0.854
    score = round(
        0.50 * (extreme_accuracy ** 1.5)
        + 0.25 * bss_normalized
        + 0.25 * uncertainty_awareness,
        4
    )
    score = max(0.0, min(1.0, score))
    # Proto3 omits zero-valued scalars from JSON
    if score == 0.0:
        score = 1e-10
    # Log detailed results for analysis
    print(f"\n{'='*60}")
    print(f"METACOGNITIVE CALIBRATION RESULTS")
    print(f"{'='*60}")
    print(f"Questions answered: {metrics['n_samples']}")
    print(f"Overall accuracy: {sum(accuracies)/len(accuracies):.2%}")
    print(f"Mean confidence: {sum(confidences)/len(confidences):.1f}%")
    print(f"Components:")
    print(f"  BSS-normalized:      {bss_normalized:.4f} (weight 0.25, BSS_raw={bss_raw:.4f})")
    print(f"  Extreme accuracy:    {extreme_accuracy:.4f} (^1.5={extreme_accuracy**1.5:.4f}, weight 0.50)")
    print(f"  Uncertainty aware:   {uncertainty_awareness:.4f} (weight 0.25)")
    print(f"  Composite score:     {score:.4f}")
    print(f"Diagnostics:")
    print(f"  Brier Skill Score (raw): {bss_raw:.4f}")
    print(f"  ECE: {metrics['ece']:.4f}")
    if conf_correct:
        print(f"  Mean conf (correct):   {np.mean(conf_correct):.1f}%")
    if conf_incorrect:
        print(f"  Mean conf (incorrect): {np.mean(conf_incorrect):.1f}%")
    print(f"\nCalibration by bin:")
    for b in metrics["bin_data"]:
        if b["count"] > 0:
            print(f"  {b['bin']}: n={b['count']}, "
                  f"conf={b['avg_confidence']:.2f}, "
                  f"acc={b['avg_accuracy']:.2f}, "
                  f"gap={b['gap']:.3f}")
    # Log per-question details
    print(f"\nPer-question results:")
    for r in results_log:
        status = "✓" if r["is_correct"] else "✗"
        print(f"  {status} [{r['confidence']:3d}%] {r['question'][:50]}... "
              f"→ {r['model_answer'][:30]}")
    return score
# ─── Run ────────────────────────────────────────────────────────────
# On Kaggle: use kbench.llm
# Locally: this will error without the Kaggle proxy, but the code is testable
if __name__ == '__main__':
    metacog_calibration.run(llm=kbench.llm)


## Cognitive Science Rationale

**Calibration** measures the correspondence between stated confidence and actual accuracy (Fischhoff, Slovic & Lichtenstein, 1977). Well-calibrated systems say "80% confident" on items they get right 80% of the time.

Systematic overconfidence (the **Dunning-Kruger effect**) is a hallmark of poor metacognition (Kruger & Dunning, 1999).


## Interpreting the Score

Score = 1 - ECE (Expected Calibration Error). A score of 1.0 means perfectly calibrated. Most LLMs show systematic overconfidence, especially on hard items.


### References
Fischhoff et al. (1977), Kruger & Dunning (1999)


## Interpreting the Score

Score = max(0, BSS) where BSS = 1 - BS/BS_ref (Brier Skill Score). BSS rewards both calibration (confidence ≈ accuracy) and resolution (high confidence on correct, low on incorrect). An always-uncertain strategy scores ~0.
